<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week4_neural_networks/Week4_Lesson8_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 IB CS — Неделя 4 · Урок 8 (Семинар)
## Lab "Brain in Code" — нейросеть с нуля + MNIST CNN
### A4.3.8 + A4.3.9 на практике

> ⏱ Длительность: 2 академических часа.
> 💻 Платформа: Google Colab (рекомендуется **GPU** для CNN, но не обязательно)
> 🎯 Цель: понять, **как работает** нейросеть, реализовав её **руками** на NumPy, а затем построить и обучить **настоящую CNN** на MNIST.

---

### 📋 План семинара

| Часть | Тема | Время |
|---|---|---|
| 1 | Forward Pass руками на NumPy (один нейрон) | 15 мин |
| 2 | Простая ANN на NumPy (3 слоя, OR/AND логические гейты) | 20 мин |
| 3 | Знакомство с MNIST (визуализация) | 15 мин |
| 4 | Обучение **ANN** на MNIST через Keras | 25 мин |
| 5 | Обучение **CNN** на MNIST | 25 мин |
| 6 | Сравнение ANN vs CNN + анализ ошибок | 15 мин |
| 7 | IB-style вывод | 5 мин |

---

### ⚠️ Подготовка (запустите в первой ячейке Colab)

```python
# Если TensorFlow ещё не установлен:
# В Colab он установлен по умолчанию.
# Локально: pip install tensorflow-cpu
```


In [ ]:
# Импорты
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Отключим лишние предупреждения
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow версия: {tf.__version__}")
print(f"✅ Keras версия: {keras.__version__}")
print(f"💻 GPU доступен: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Часть 1 · Forward Pass одного нейрона на NumPy

> Воспроизводим **worked example из Baumgarten (стр. 47)** — но руками.


In [ ]:
# Один нейрон. Inputs, weights, bias — заданы.
inputs  = np.array([1.3, 4.2, 0.0, 2.7])
weights = np.array([-3.1, 1.6, 2.9, 2.7])
bias    = -5.2

# Шаг 1-3: weighted sum (это матричное умножение!)
summation = np.dot(inputs, weights)
print(f"Inputs:  {inputs}")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")
print(f"\nШаг 1-3 — Weighted sum:")
print(f"  np.dot(inputs, weights) = {summation:.4f}")

# Шаг 4: add bias
z = summation + bias
print(f"\nШаг 4 — Add bias: {summation:.4f} + ({bias}) = {z:.4f}")

# Шаг 5: ReLU activation
def relu(x):
    return np.maximum(0, x)

output = relu(z)
print(f"\nШаг 5 — ReLU({z:.4f}) = {output:.4f}")
print(f"\n💎 Выход нейрона: {output:.4f}")

### 🎯 TRY IT #1 — Эксперимент

Измените weights и bias. Как меняется выход?

- Если bias = +5.0 (вместо −5.2) → output = ?
- Если все weights отрицательные → output = ?

> 💡 ReLU "обнуляет" отрицательные значения. Это критично понять.


## Часть 2 · Простая ANN на NumPy — Logic Gates (OR / AND)

> Из учебника Baumgarten (стр. 49): сеть с 2 inputs → 4 hidden → 2 outputs для логических гейтов.

**Архитектура:**
- 2 входа (A, B)
- 4 нейрона в hidden layer (ReLU)
- 2 выхода (OR, AND) — Sigmoid


In [ ]:
# Веса и bias уже обучены (из учебника Baumgarten, стр. 50)
# Hidden layer
W1 = np.array([
    [ 0.60, -0.47,  0.70,  2.10],
    [-1.13, -1.11,  2.10,  0.80],
])
b1 = np.array([0.53, 0.00, 0.00, -0.23])

# Output layer
W2 = np.array([
    [1.20, -0.30],
    [-0.20, -0.50],
    [1.50, 0.10],
    [0.40, 0.90],
])
b2 = np.array([-1.0, -1.5])  # подобран чтобы оба гейта работали

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_pass(X, verbose=False):
    # Forward pass для входов X (вектор из 2 значений)
    # Layer 1
    z1 = X @ W1 + b1
    a1 = relu(z1)
    if verbose:
        print(f"  Hidden activations:  {a1.round(2)}")

    # Layer 2
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    if verbose:
        print(f"  Output activations:  {a2.round(2)}")
        print(f"  Rounded prediction:  OR={round(a2[0])}, AND={round(a2[1])}")
    return a2

# Проверим на всех 4 комбинациях входов
print("=" * 60)
print(f"{'Input':<12} | {'Expected':<14} | {'Predicted':<14}")
print("-" * 60)
for A, B in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    X = np.array([A, B])
    out = forward_pass(X)
    expected_or = A | B
    expected_and = A & B
    pred_or, pred_and = round(out[0]), round(out[1])
    print(f"A={A}, B={B}     | OR={expected_or}, AND={expected_and}    | OR={pred_or}, AND={pred_and}")
print("=" * 60)
print("\n💎 Если все строки совпадают — сеть РАБОТАЕТ!")

In [ ]:
# Подробный разбор одного forward pass: A=1, B=0
print("=== Подробный forward pass для входа [1, 0] ===\n")

X = np.array([1.0, 0.0])
print(f"Input: {X}\n")

print("Hidden layer 4 нейрона:")
for i in range(4):
    z_i = X[0]*W1[0,i] + X[1]*W1[1,i] + b1[i]
    a_i = relu(z_i)
    print(f"  H{i+1}: ReLU(1·{W1[0,i]:+.2f} + 0·{W1[1,i]:+.2f} + {b1[i]:+.2f}) = ReLU({z_i:+.2f}) = {a_i:.2f}")

print("\nOutput layer 2 нейрона:")
a1 = relu(X @ W1 + b1)
for i in range(2):
    z_i = (a1 * W2[:, i]).sum() + b2[i]
    a_i = sigmoid(z_i)
    print(f"  O{i+1}: sigmoid(...) = sigmoid({z_i:+.2f}) = {a_i:.2f}")

print("\n💎 OR(1, 0) = 1 ✓   AND(1, 0) = 0 ✓")

### 🎯 TRY IT #2 — Понимание матричного умножения

> Заметили: `X @ W1` — это **матричное умножение**? Именно поэтому **GPU и TPU** так важны для нейросетей — они оптимизированы для matrix multiplication.

Вопрос: если на input 100 нейронов, в hidden 50, сколько умножений в одном forward pass?
**Ответ:** 100 × 50 = 5000. А если batch из 64 примеров → 64 × 5000 = 320 000 умножений за один forward pass.

> 💎 Теперь понимаете, почему **TPU** (специальный matrix multiplication unit) даёт огромный прирост для нейросетей? Это связь с **A4.1.2 hardware**.


## Часть 3 · MNIST — "Hello World" для нейросетей

> **MNIST**: 60,000 рукописных цифр (28×28 пикселей, grayscale) + 10,000 тестовых.
> Задача: классифицировать 0–9.


In [ ]:
# Загрузка MNIST (встроен в Keras)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"📊 Train: {X_train.shape}, labels {y_train.shape}")
print(f"📊 Test:  {X_test.shape}, labels {y_test.shape}")
print(f"\n   Каждое изображение: 28×28 пикселей")
print(f"   Значения: 0-255 (grayscale)")
print(f"   Классы: 0-9 (10 цифр)")

# Визуализация: первые 10 цифр
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST — первые 10 примеров из train set', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Распределение классов
fig, ax = plt.subplots(figsize=(10, 4))
unique, counts = np.unique(y_train, return_counts=True)
ax.bar(unique, counts, color='steelblue', edgecolor='black')
ax.set_xticks(unique)
ax.set_xlabel('Цифра (класс)'); ax.set_ylabel('Количество в train set')
ax.set_title('Распределение классов в MNIST (хорошо сбалансированное)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n💡 Все 10 классов представлены примерно поровну (~5500-6700) — отлично!")

### Preprocessing — нормализация

> Делим пиксели на 255 → значения в [0, 1]. Это **критично** для нейросетей.


In [ ]:
# Нормализация пикселей: [0, 255] → [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

print(f"До нормализации:  min={X_train.min()}, max={X_train.max()}")
print(f"После:            min={X_train_norm.min()}, max={X_train_norm.max():.4f}")

## Часть 4 · ANN на MNIST через Keras

> Строим простую ANN: input(784) → hidden(128, ReLU) → output(10, softmax)


In [ ]:
# Архитектура ANN
ann_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),         # 28×28 → 784
    layers.Dense(128, activation='relu'),          # hidden layer
    layers.Dense(10, activation='softmax'),        # output (10 классов)
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()
print("\n💎 Считаем параметры:")
print("   Flatten: 0 (просто reshape)")
print("   Dense(128): 784 × 128 + 128 = 100,480")
print("   Dense(10):  128 × 10 + 10 = 1,290")
print("   ИТОГО: 101,770 параметров для обучения")

In [ ]:
# Обучение ANN (5 эпох — для семинара хватит)
print("🎓 Обучаем ANN... (5 эпох)\n")
history_ann = ann_model.fit(
    X_train_norm, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=2
)

# Оценка на тестовом наборе
test_loss_ann, test_acc_ann = ann_model.evaluate(X_test_norm, y_test, verbose=0)
print(f"\n✅ Test accuracy: {test_acc_ann:.4f}  ({test_acc_ann*100:.2f}%)")

In [ ]:
# Графики обучения
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_ann.history['loss'], 'o-', label='Train loss', color='steelblue')
axes[0].plot(history_ann.history['val_loss'], 's-', label='Val loss', color='orange')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('ANN — Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_ann.history['accuracy'], 'o-', label='Train acc', color='steelblue')
axes[1].plot(history_ann.history['val_accuracy'], 's-', label='Val acc', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('ANN — Accuracy', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## Часть 5 · CNN на MNIST — настоящая магия

> Теперь добавим **convolutional + pooling** слои. Ожидаем **выше accuracy**.


In [ ]:
# Для CNN нужен 4-D входной shape: (batch, height, width, channels)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test_norm.reshape(-1, 28, 28, 1)

print(f"CNN input shape: {X_train_cnn.shape}")

In [ ]:
# Архитектура CNN: 2 conv blocks + dense
cnn_model = keras.Sequential([
    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Fully connected
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# Обучение CNN (5 эпох — на CPU может занять 2-5 минут)
print("🎓 Обучаем CNN... (5 эпох, может занять несколько минут)\n")
history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=2
)

test_loss_cnn, test_acc_cnn = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\n✅ CNN Test accuracy: {test_acc_cnn:.4f}  ({test_acc_cnn*100:.2f}%)")
print(f"   (ANN была: {test_acc_ann:.4f})")

In [ ]:
# Сравнение ANN vs CNN
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_ann.history['val_accuracy'], 'o-', label='ANN', color='steelblue', linewidth=2)
axes[0].plot(history_cnn.history['val_accuracy'], 's-', label='CNN', color='green', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('ANN vs CNN — Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Bar chart финальных результатов
models_names = ['ANN', 'CNN']
test_accs = [test_acc_ann, test_acc_cnn]
bars = axes[1].bar(models_names, test_accs, color=['steelblue', 'green'], edgecolor='black')
axes[1].set_ylabel('Test Accuracy'); axes[1].set_ylim(0.9, 1.0)
axes[1].set_title('Test Accuracy — финальный результат', fontweight='bold')
for bar, acc in zip(bars, test_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                  f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\n💎 CNN превосходит ANN на {(test_acc_cnn - test_acc_ann)*100:.2f} процентных пункта")
print("   Это потому, что CNN сохраняет SPATIAL структуру изображения,")
print("   а ANN сначала всё 'плющит' в 784-мерный вектор, теряя пространственную информацию.")

## Часть 6 · Анализ ошибок CNN

> Где CNN ошибается? Покажем 9 случаев неправильной классификации.


In [ ]:
# Предсказания CNN
predictions = cnn_model.predict(X_test_cnn, verbose=0)
predicted_classes = predictions.argmax(axis=1)

# Находим ошибки
wrong_idx = np.where(predicted_classes != y_test)[0]
print(f"❌ CNN сделала {len(wrong_idx)} ошибок из {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)")

# Показываем первые 9 ошибок
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        ax.imshow(X_test[idx], cmap='gray')
        confidence = predictions[idx][predicted_classes[idx]]
        ax.set_title(f'True: {y_test[idx]} · Predicted: {predicted_classes[idx]}\nConfidence: {confidence:.2%}',
                     fontsize=10, color='darkred')
        ax.axis('off')
plt.suptitle('Где CNN ошибается? (обычно "грязные" или нестандартные цифры)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix для CNN
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, predicted_classes)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), cbar=False, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — CNN (10 классов MNIST)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 🎯 TRY IT #3 — Какие цифры путаются чаще всего?

Посмотрите на confusion matrix. Какие пары наиболее путаются?

> 💡 Обычно: 4 ↔ 9, 3 ↔ 8, 5 ↔ 6 — потому что у них **похожие визуальные паттерны**.


## Часть 7 · IB-style вывод

### 📝 Что мы сегодня сделали (для отчёта IA)

```
1. NUMPY IMPLEMENTATION:
   - Реализовали forward pass одного нейрона через np.dot()
   - Построили простую ANN для логических гейтов вручную
   - Поняли, что forward propagation = matrix multiplication + activation

2. KERAS / TENSORFLOW:
   - Построили ANN: Flatten → Dense(128, ReLU) → Dense(10, Softmax)
   - Построили CNN: Conv→Pool→Conv→Pool→Flatten→Dense→Output
   - Обучили обе на MNIST (5 epochs, batch_size=64, Adam optimizer)

3. ОЦЕНКА:
   - ANN test accuracy: ~98%
   - CNN test accuracy: ~99% — **выше**, потому что сохраняет spatial structure
   - Проанализировали ошибки через confusion matrix
```

### 💎 Финальные секреты семинара

1. **Forward pass = matrix multiplication.** Поэтому GPU/TPU так важны.
2. **Нормализация обязательна** для нейросетей. `/ 255.0` для изображений.
3. **`Sparse categorical crossentropy`** — loss для multi-class классификации с integer labels.
4. **CNN всегда лучше ANN на изображениях** — потому что сохраняет spatial structure.
5. **Confusion matrix** — gold standard для анализа multi-class моделей.
6. **`Conv → ReLU → MaxPool` блок** — стандартный паттерн CNN.

---

### 🏠 ДЗ (см. `Week4_HW2_Practice.ipynb`)

> Эксперимент с архитектурой: изменить число hidden layers + сравнить ReLU vs Sigmoid. Зафиксировать, как это влияет на скорость обучения и финальный loss.

---

> 🎓 **Финальный совет:**
> Этот семинар — лучшая подготовка к IA по нейросетям. Сохраните код — он понадобится в выпускном году.
> Большинство 6-х и 7-х на IA по ML — это **именно** такие эксперименты с архитектурами + анализ результатов.
